In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42


In [6]:
def preprocess(df, target_col, drop_cols=None):
    df = df.copy()
    if drop_cols:
        df = df.drop(columns=drop_cols, errors='ignore')
    for col in df.columns:
        if df[col].dtype == 'object' or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].astype(object).fillna('Unknown')
        else:
            df[col] = df[col].fillna(df[col].median())
    y = df[target_col]
    if y.dtype == 'object' or pd.api.types.is_string_dtype(y):
        y = y.map(lambda v: 1 if str(v).strip().upper() in ['Y','YES','1','TRUE','FRAUD'] else 0)
    X = df.drop(columns=[target_col])
    for col in X.columns:
        if X[col].dtype == 'object' or pd.api.types.is_string_dtype(X[col]):
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
        elif np.issubdtype(X[col].dtype, np.datetime64):
            X[col] = pd.to_datetime(X[col]).astype('int64') // 10**9
    return X, y

def run_models(X, y, name):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    results = {}

    rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=RANDOM_STATE)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_proba = rf.predict_proba(X_test)[:,1]

    results['RandomForest'] = {
        'precision': precision_score(y_test, rf_pred),
        'recall': recall_score(y_test, rf_pred),
        'f1': f1_score(y_test, rf_pred),
        'auc': roc_auc_score(y_test, rf_proba),
        'confusion_matrix': confusion_matrix(y_test, rf_pred).tolist(),
        'top_features': sorted(zip(X.columns, rf.feature_importances_), key=lambda x: -x[1])[:5]
    }

    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)
    mlp = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=1000, random_state=RANDOM_STATE, early_stopping=True)
    mlp.fit(X_train_bal, y_train_bal)
    mlp_pred = mlp.predict(X_test_scaled)
    mlp_proba = mlp.predict_proba(X_test_scaled)[:,1]

    results['NeuralNetwork'] = {
        'precision': precision_score(y_test, mlp_pred),
        'recall': recall_score(y_test, mlp_pred),
        'f1': f1_score(y_test, mlp_pred),
        'auc': roc_auc_score(y_test, mlp_proba),
        'confusion_matrix': confusion_matrix(y_test, mlp_pred).tolist()
    }

    print(f"\n{'='*50}\n{name}\n{'='*50}")
    for model_name, metrics in results.items():
        print(f"\n--- {model_name} ---")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall:    {metrics['recall']:.4f}")
        print(f"F1-score:  {metrics['f1']:.4f}")
        print(f"AUC:       {metrics['auc']:.4f}")
        print(f"Confusion Matrix: {metrics['confusion_matrix']}")
        if 'top_features' in metrics:
            print(f"Top 5 features: {[f[0] for f in metrics['top_features']]}")

    return results


In [7]:

# ===== AUTO INSURANCE =====
auto = pd.read_excel('auto_insurance_fraud.xlsx', sheet_name='Fraud_Detection_decsion tree')
X_auto, y_auto = preprocess(auto, target_col='fraud_reported',
                              drop_cols=['policy_number', 'incident_location', 'policy_bind_date', 'incident_date'])
auto_results = run_models(X_auto, y_auto, "AUTO INSURANCE FRAUD DATASET")


AUTO INSURANCE FRAUD DATASET

--- RandomForest ---
Precision: 0.6111
Recall:    0.7097
F1-score:  0.6567
AUC:       0.8201
Confusion Matrix: [[160, 28], [18, 44]]
Top 5 features: ['incident_severity', 'insured_hobbies', 'total_claim_amount', 'vehicle_claim', 'insured_zip']

--- NeuralNetwork ---
Precision: 0.4928
Recall:    0.5484
F1-score:  0.5191
AUC:       0.7385
Confusion Matrix: [[153, 35], [28, 34]]


In [8]:
# ===== HEALTHCARE =====
health = pd.read_csv('healthcare_fraud_detection.csv')
X_health, y_health = preprocess(health, target_col='Is_Fraud',
                                  drop_cols=['Claim_ID', 'Claim_Submission_Date'])
health_results = run_models(X_health, y_health, "HEALTHCARE FRAUD DATASET")


HEALTHCARE FRAUD DATASET

--- RandomForest ---
Precision: 0.6393
Recall:    0.9420
F1-score:  0.7617
AUC:       0.9900
Confusion Matrix: [[2183, 110], [12, 195]]
Top 5 features: ['Days_Between_Service_and_Claim', 'Claim_Status', 'Claim_Amount', 'Approved_Amount', 'Number_of_Claims_Per_Provider_Monthly']

--- NeuralNetwork ---
Precision: 0.9188
Recall:    0.8744
F1-score:  0.8960
AUC:       0.9953
Confusion Matrix: [[2277, 16], [26, 181]]
